# Create resized NYU Depth v2 H5 dataset for METER on Kaggle

This notebook converts an origin_data-style NYU H5 dataset into pre-resized 192 x 256 H5 files so METER training can run with resize_inputs=False.

- RGB: bilinear resize
- Depth: nearest resize
- Depth unit is preserved as meters
- Folder layout is preserved
- Fast shallow file discovery is used: train/*/*.h5 and val/*.h5
- Corrupt pseudo-H5 files are skipped and written to the summary
- Output goes to /kaggle/working/nyu_depth_v2_resized_192x256 by default


In [ ]:
from __future__ import annotations

import json
import shutil
from concurrent.futures import ProcessPoolExecutor, as_completed
from dataclasses import asdict, dataclass
from pathlib import Path

import cv2
import h5py
import numpy as np
from tqdm.auto import tqdm


In [ ]:
@dataclass(frozen=True)
class Config:
    # Set this to your Kaggle dataset slug folder under /kaggle/input.
    # Example: if files are in /kaggle/input/nyu-origin-data/origin_data, use "nyu-origin-data".
    kaggle_dataset_slug: str = "your-dataset-slug"

    # If None, notebook searches common candidates under /kaggle/input/{slug}.
    input_root: str | None = None
    output_root: str = "/kaggle/working/nyu_depth_v2_resized_192x256"

    height: int = 192
    width: int = 256
    num_workers: int = 4
    limit_per_split: int | None = None
    overwrite: bool = True

    # Fastest: None. Smaller but slower: "lzf". Avoid gzip for speed.
    compression: str | None = None

    # Make a zip at the end. Can be slow and large; disable if you plan to save folder output directly.
    create_zip: bool = False
    zip_path: str = "/kaggle/working/nyu_depth_v2_resized_192x256.zip"


CFG = Config()
print(json.dumps(asdict(CFG), indent=2))


In [ ]:
def resolve_input_root(cfg: Config) -> Path:
    if cfg.input_root is not None:
        root = Path(cfg.input_root)
        if (root / "train").exists() and (root / "val").exists():
            return root
        raise FileNotFoundError(f"input_root does not contain train/val: {root}")

    base = Path("/kaggle/input") / cfg.kaggle_dataset_slug
    candidates = [
        base / "origin_data",
        base / "nyu_depth_v2",
        base / "nyu-depth-v2",
        base,
    ]
    for root in candidates:
        if (root / "train").exists() and (root / "val").exists():
            return root
    raise FileNotFoundError("Could not find train/val. Checked: " + ", ".join(map(str, candidates)))


INPUT_ROOT = resolve_input_root(CFG)
OUTPUT_ROOT = Path(CFG.output_root)
print("INPUT_ROOT =", INPUT_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)


In [ ]:
def discover_files(input_root: Path, split: str, limit: int | None) -> list[Path]:
    # Fast path for the expected Kaggle layout:
    #   train/<scene_name>/*.h5
    #   val/*.h5
    # This avoids a slow recursive rglob over the entire dataset before processing starts.
    split_root = input_root / split
    if split == "train":
        files = sorted(path for scene_dir in split_root.iterdir() if scene_dir.is_dir() for path in scene_dir.glob("*.h5"))
    elif split == "val":
        files = sorted(split_root.glob("*.h5"))
    else:
        raise ValueError(f"Unsupported split: {split}")
    return files[:limit] if limit is not None else files


for split in ["train", "val"]:
    files = discover_files(INPUT_ROOT, split, CFG.limit_per_split)
    print(split, len(files), "files")


In [ ]:
def resize_one_file(task: tuple[str, str, str, int, int, str | None, bool]) -> dict:
    source_text, split_root_text, output_root_text, height, width, compression, overwrite = task
    source_path = Path(source_text)
    split_root = Path(split_root_text)
    output_root = Path(output_root_text)
    relative_path = source_path.relative_to(split_root)
    target_path = output_root / relative_path

    if target_path.exists() and not overwrite:
        return {"status": "skipped_existing", "source": source_text, "target": str(target_path)}

    try:
        with h5py.File(source_path, "r") as source:
            if "rgb" not in source or "depth" not in source:
                raise KeyError("missing rgb/depth datasets")

            rgb = source["rgb"][:]
            depth = source["depth"][:]

            if rgb.ndim != 3 or rgb.shape[0] != 3:
                raise ValueError(f"expected rgb (3,H,W), got {rgb.shape}")
            if depth.ndim != 2:
                raise ValueError(f"expected depth (H,W), got {depth.shape}")

            rgb_hwc = np.transpose(rgb, (1, 2, 0))
            rgb_resized = cv2.resize(rgb_hwc, (width, height), interpolation=cv2.INTER_LINEAR)
            rgb_resized = np.transpose(rgb_resized, (2, 0, 1)).astype(rgb.dtype, copy=False)
            depth_resized = cv2.resize(depth, (width, height), interpolation=cv2.INTER_NEAREST).astype(depth.dtype, copy=False)

            target_path.parent.mkdir(parents=True, exist_ok=True)
            temp_path = target_path.with_suffix(target_path.suffix + ".tmp")
            if temp_path.exists():
                temp_path.unlink()

            dataset_kwargs = {} if compression is None else {"compression": compression}
            with h5py.File(temp_path, "w") as target:
                target.create_dataset("rgb", data=rgb_resized, **dataset_kwargs)
                target.create_dataset("depth", data=depth_resized, **dataset_kwargs)

                for key, value in source.attrs.items():
                    try:
                        target.attrs[key] = value
                    except TypeError:
                        target.attrs[key] = str(value)

                for key in source.keys():
                    if key in {"rgb", "depth"}:
                        continue
                    try:
                        source.copy(key, target)
                    except Exception:
                        target.attrs[f"skipped_dataset_{key}"] = "copy failed during resize export"

            temp_path.replace(target_path)
            return {"status": "written", "source": source_text, "target": str(target_path)}
    except Exception as error:
        return {
            "status": "failed",
            "source": source_text,
            "target": str(target_path),
            "error": repr(error),
        }


In [ ]:
def prepare_output(output_root: Path, overwrite: bool) -> None:
    if output_root.exists() and overwrite:
        shutil.rmtree(output_root)
    output_root.mkdir(parents=True, exist_ok=True)


def run_split(split: str) -> dict:
    split_root = INPUT_ROOT / split
    out_split_root = OUTPUT_ROOT / split
    files = discover_files(INPUT_ROOT, split, CFG.limit_per_split)
    out_split_root.mkdir(parents=True, exist_ok=True)

    tasks = [
        (str(path), str(split_root), str(out_split_root), CFG.height, CFG.width, CFG.compression, CFG.overwrite)
        for path in files
    ]
    counts = {"total": len(tasks), "written": 0, "skipped_existing": 0, "failed": 0}
    failures = []

    with ProcessPoolExecutor(max_workers=CFG.num_workers) as executor:
        futures = [executor.submit(resize_one_file, task) for task in tasks]
        for future in tqdm(as_completed(futures), total=len(futures), desc=split):
            result = future.result()
            status = result["status"]
            counts[status] = counts.get(status, 0) + 1
            if status == "failed":
                failures.append(result)
    counts["failures"] = failures
    return counts


prepare_output(OUTPUT_ROOT, CFG.overwrite)
summary = {
    "input_root": str(INPUT_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "height": CFG.height,
    "width": CFG.width,
    "compression": CFG.compression,
    "splits": {},
}
for split in ["train", "val"]:
    summary["splits"][split] = run_split(split)

summary_path = OUTPUT_ROOT / "resize_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2)[:5000])
print("summary_path =", summary_path)


In [ ]:
def verify_some_outputs(output_root: Path) -> None:
    for split in ["train", "val"]:
        files = sorted((output_root / split).rglob("*.h5"))
        print(split, "output files:", len(files))
        for path in files[:3]:
            with h5py.File(path, "r") as h5:
                print(path.relative_to(output_root), h5["rgb"].shape, h5["rgb"].dtype, h5["depth"].shape, h5["depth"].dtype)


verify_some_outputs(OUTPUT_ROOT)


In [ ]:
if CFG.create_zip:
    zip_base = Path(CFG.zip_path).with_suffix("")
    archive = shutil.make_archive(str(zip_base), "zip", root_dir=OUTPUT_ROOT)
    print("created zip:", archive)
else:
    print("Zip disabled. To save across Kaggle sessions, create a Kaggle Dataset from OUTPUT_ROOT or set create_zip=True.")
